<a href="https://colab.research.google.com/github/jceltruda/Projects-in-AI-and-ML/blob/main/Project_6/ML_AI_Projects_6_Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2

### RAG (Retrieval-augmented generation) vs. No RAG Question Answering

I will evaluate model responses on recent NHL standings/playoffs with and without RAG.

Sources:
* https://proceedings.neurips.cc/paper/2020/file/6b493230205f780e1bc26945df7481e5-Paper.pdf
* https://www.geeksforgeeks.org/artificial-intelligence/rag-with-langchain/

Problem and method:
* RAG addresses a number of issues or quirks with LLM systems. Modern LLMs are very intelligent, but lack knowledge past their training date. In addition, smaller models may not have relevant world data "remembered" in their parameters. Another caveat, knowledge bases may be too large for a LLM to effectively ingest and decipher. RAG solves these issues by retrieving context relevant to the user prompt and passing it to the LLM along with the prompt.
* This is implemented by chunking and embedding input document context. A RAG system automatically fetches context relating to the user prompt based on similarity (meaning) in the embedding space.

Limitations or challenges:
* Knowledge base management: One challenge is maintaining an updated knowledge base for real-time tasks. In the example below, I query NHL playoff qualifications, which change daily. RAG systems for live use cases like this require constant management of the information in the knowledge base.
* Context fragmentation: Another challenge is maintaining relationships between different segments of a document. Documents are split into distinct chunks which are separetly embedded, which can cause relationships between different segments of context to be lost, for example a variable definition early in a research paper will not be present when the variable appears in an equation later on.

### RAG Demo


In [ ]:
!pip install langchain langchain-community langchain-anthropic langchain-huggingface sentence-transformers faiss-cpu beautifulsoup4 pandas python-dotenv

In [3]:
import os

os.environ["USER_AGENT"] = "MyUniversityRAGProject/1.0" # To identify scraper for Wikipedia

import pandas as pd
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Read API key
api_file = "api_key.txt"
with open(api_file, "r") as f:
    api_key = f.read().strip()
os.environ["ANTHROPIC_API_KEY"] = api_key

In [ ]:
# Fetch and parse wikipedia article for RAG
url = "https://en.wikipedia.org/wiki/2026_Stanley_Cup_playoffs"
loader = WebBaseLoader(url)
docs = loader.load()
print(f"Loaded document from Wikipedia.")

# Split document into chunks of size 500 with 50 overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)
print(f"Document split into {len(splits)} chunks.")

# Vector Database creation
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# FAISS vector store
vectorstore = FAISS.from_documents(documents=splits, embedding=embedding_model)
print("FAISS vector database created")

In [5]:
# RAG testing questions
questions = [
    "Who holds the broadcast rights for the 2026 Stanley Cup Final in the United States?",
    "Which team won the Presidents' Trophy in the 2025-2026 regular season?",
    "Did the Buffalo Sabres make the playoffs?",
    "Which team ended their long standing playoff streak?",
    "On what date do the playoffs begin?",
    "Which Metropolitan teams have qualified for the playoffs?"
]

# Using Claude 3 Haiku
llm = ChatAnthropic(model_name="claude-3-haiku-20240307", temperature=0)

# No RAG prompt (same as RAG prompt aside from adding context)
baseline_system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Answer the question based only on your internal knowledge. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
)

baseline_prompt = ChatPromptTemplate.from_messages([
    ("system", baseline_system_prompt),
    ("human", "{question}"),
])

# Connect prompt to model
baseline_chain = baseline_prompt | llm

results = []

print("Running No-RAG pipeline")
for q in questions:
    response = baseline_chain.invoke({"question": q})
    # Store the AIMessage text content
    results.append({
        "Question": q,
        "No_RAG_Answer": response.content
    })

# Fetch the top 3 relevant chunks for RAG
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "Context:\n{context}"
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Format the retrieved documents into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Running RAG pipeline")
for i, q in enumerate(questions):
    response = rag_chain.invoke(q)

    # Store the final generated RAG answer
    results[i]["RAG_Answer"] = response

# Format and save results
df = pd.DataFrame(results)
output_file = "results.csv"
df.to_csv(output_file, index=False)

print(f"Evaluation complete")

Running No-RAG pipeline
Running RAG pipeline
Evaluation complete


#### Results

In [7]:
import pandas as pd

# Read experiment results
results_df = pd.read_csv('results.csv')

# Display results
for index, row in results_df.iterrows():
    print(f"Question {index + 1}:")
    print(row['Question'])
    print("\nNo RAG response:")
    print(row['No_RAG_Answer'])
    print("\nRag response:")
    print(row['RAG_Answer'])
    print('\n')

Question 1:
Who holds the broadcast rights for the 2026 Stanley Cup Final in the United States?

No RAG response:
I do not know who holds the broadcast rights for the 2026 Stanley Cup Final in the United States.

Rag response:
According to the context, the 2026 Stanley Cup Final will be broadcast by ABC in the United States.


Question 2:
Which team won the Presidents' Trophy in the 2025-2026 regular season?

No RAG response:
I'm afraid I don't have any information about the winner of the Presidents' Trophy in the 2025-2026 NHL regular season. As an AI assistant, I don't have access to future sports results or predictions.

Rag response:
According to the context provided, the Colorado Avalanche will qualify for the playoffs as the Presidents' Trophy winners with the most points (i.e. best record) during the 2025-2026 regular season.


Question 3:
Did the Buffalo Sabres make the playoffs?

No RAG response:
No, the Buffalo Sabres did not make the playoffs this season. They finished the 2

Analysis:
* As shown above, the RAG based approach correctly pulled relevant information from the knowledge base and correclty answered all questions.
* The Non-RAG approach failed to correctly answer a single question as the questions were ambiguous and it didn't have updated information.

Potential extension of project:
* Implement hierarchical retrival to limitation of context fragmentation. This involves storing each document on multiple levels: document, section, and chunk. This allows the retriever to first identify relevant high level sections and then narrow its search down to the most relevant chunks inside those sections, helping to preserve relationships between chunks from the same source.